# 01 · Transformación y limpieza de los datos

**Proyecto EDA – Campañas de marketing de un banco portugués**

Objetivo de este notebook: dejar los dos conjuntos de datos (`bank-additional.csv` y `customer-details.xlsx`) limpios, con tipos correctos, sin errores de formato y unidos en un único dataset listo para el análisis.

Estructura:
1. Carga de los datos en bruto
2. Diagnóstico de calidad (tipos, nulos, duplicados, formatos)
3. Limpieza del dataset de campañas
4. Limpieza del dataset de clientes
5. Unión de ambos datasets
6. Optimización de memoria y exportación



In [8]:
# Preparación para Google Colab (en local no hace nada)
import os
import sys
import zipfile

if "google.colab" in sys.modules:
    proyecto = "/content/eda-bank-marketing"
    if not os.path.exists(proyecto):
        from google.colab import files
        print("Selecciona el archivo eda-bank-marketing.zip")
        subido = files.upload()
        zipfile.ZipFile(next(iter(subido)), "r").extractall("/content")
    os.chdir(proyecto + "/notebooks")
    print("Listo. Carpeta de trabajo:", os.getcwd())

Listo. Carpeta de trabajo: /content/eda-bank-marketing/notebooks


In [9]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.append(str(Path.cwd().parent / "src"))
import funciones as fn

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

## 1. Carga de los datos en bruto

In [10]:
bank_raw = fn.cargar_bank()
print(f"bank-additional.csv -> {bank_raw.shape[0]:,} filas x {bank_raw.shape[1]} columnas")
bank_raw.head()

bank-additional.csv -> 43,000 filas x 24 columnas


,Unnamed: 0,age,job,marital,education,default,housing,loan,contact,duration,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y,date,latitude,longitude,id_
0,0,NaN,housemaid,MARRIED,basic.4y,0.0,0.0,0.0,telephone,261,1,999,0,NONEXISTENT,1.1,"93,994","-36,4","4,857",5191,no,2-agosto-2019,41.495,-71.233,089b39d8-e4d0-461b-87d4-814d71e0e079
1,1,57.0,services,MARRIED,high.school,NaN,0.0,0.0,telephone,149,1,999,0,NONEXISTENT,1.1,"93,994","-36,4",NaN,5191,no,14-septiembre-2016,34.601,-83.923,e9d37224-cb6f-4942-98d7-46672963d097
2,2,37.0,services,MARRIED,high.school,0.0,1.0,0.0,telephone,226,1,999,0,NONEXISTENT,1.1,"93,994","-36,4","4,857",5191,no,15-febrero-2019,34.939,-94.847,3f9f49b5-e410-4948-bf6e-f9244f04918b
3,3,40.0,admin.,MARRIED,basic.6y,0.0,0.0,0.0,telephone,151,1,999,0,NONEXISTENT,1.1,"93,994","-36,4",NaN,5191,no,29-noviembre-2015,49.041,-70.308,9991fafb-4447-451a-8be2-b0df6098d13e
4,4,56.0,services,MARRIED,high.school,0.0,0.0,1.0,telephone,307,1,999,0,NONEXISTENT,1.1,"93,994","-36,4",NaN,5191,no,29-enero-2017,38.033,-104.463,eca60b76-70b6-4077-80ba-bc52e8ebb0eb


El Excel tiene **3 hojas** (una por año de alta del cliente). `pd.read_excel(sheet_name=None)` devuelve un diccionario `{hoja: DataFrame}` que se apila con `pd.concat`, conservando el año de la hoja en una nueva columna `anio_alta`.

In [11]:
hojas = pd.read_excel(fn.RUTA_RAW / "customer-details.xlsx", sheet_name=None)
print({nombre: hoja.shape for nombre, hoja in hojas.items()})

customers_raw = fn.cargar_customers()
print(f"customer-details.xlsx (3 hojas apiladas) -> {customers_raw.shape[0]:,} filas x {customers_raw.shape[1]} columnas")
customers_raw.head()

{'2012': (20115, 7), '2013': (8965, 7), '2014': (14090, 7)}
customer-details.xlsx (3 hojas apiladas) -> 43,170 filas x 8 columnas


,Unnamed: 0,Income,Kidhome,Teenhome,Dt_Customer,NumWebVisitsMonth,ID,anio_alta
0,0,161770,1,0,2012-04-04,29,089b39d8-e4d0-461b-87d4-814d71e0e079,2012
1,1,85477,1,1,2012-12-30,7,e9d37224-cb6f-4942-98d7-46672963d097,2012
2,2,147233,1,1,2012-02-02,5,3f9f49b5-e410-4948-bf6e-f9244f04918b,2012
3,3,121393,1,2,2012-12-21,29,9991fafb-4447-451a-8be2-b0df6098d13e,2012
4,4,63164,1,2,2012-06-20,20,eca60b76-70b6-4077-80ba-bc52e8ebb0eb,2012


## 2. Diagnóstico de calidad

### 2.1 Dataset de campañas

In [12]:
bank_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43000 entries, 0 to 42999
Data columns (total 24 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Unnamed: 0      43000 non-null  int64  
 1   age             37880 non-null  float64
 2   job             42655 non-null  object 
 3   marital         42915 non-null  object 
 4   education       41193 non-null  object 
 5   default         34019 non-null  float64
 6   housing         41974 non-null  float64
 7   loan            41974 non-null  float64
 8   contact         43000 non-null  object 
 9   duration        43000 non-null  int64  
 10  campaign        43000 non-null  int64  
 11  pdays           43000 non-null  int64  
 12  previous        43000 non-null  int64  
 13  poutcome        43000 non-null  object 
 14  emp.var.rate    43000 non-null  float64
 15  cons.price.idx  42529 non-null  object 
 16  cons.conf.idx   43000 non-null  object 
 17  euribor3m       33744 non-null 

In [6]:
fn.resumen_calidad(bank_raw)

,tipo,nulos,pct_nulos,valores_unicos
euribor3m,object,9256,21.53,309
default,float64,8981,20.89,2
age,float64,5120,11.91,78
education,object,1807,4.20,7
loan,float64,1026,2.39,2
housing,float64,1026,2.39,2
cons.price.idx,object,471,1.10,26
job,object,345,0.80,11
date,object,248,0.58,1825
marital,object,85,0.20,3


**Problemas detectados:**

| Problema | Columnas | Decisión |
|---|---|---|
| Columna índice sobrante (`Unnamed: 0`) | — | Eliminar |
| Nombres con puntos (`emp.var.rate`) y guion final (`id_`) | varias | Pasar a *snake_case* |
| Números guardados como texto con **coma decimal** (`"93,994"`) | `cons.price.idx`, `cons.conf.idx`, `euribor3m`, `nr.employed` | Convertir a `float` |
| Mayúsculas inconsistentes (`MARRIED`, `NONEXISTENT`) y puntos en categorías (`admin.`, `basic.4y`) | `marital`, `poutcome`, `job`, `education` | Minúsculas y `_` |
| Fecha en texto con el mes en español (`2-agosto-2019`) | `date` | Parsear a `datetime` |
| Faltan `contact_month` y `contact_year` (están en el enunciado) | — | Derivarlas de `date` |
| Nulos | `age`, `euribor3m`, `default`, `education`… | Ver 3.2 |
| `pdays = 999` significa "nunca contactado", no 999 días | `pdays` | Pasar a `NaN` + indicador |

In [7]:
# Valores de las variables categóricas: se ven las mayúsculas y los puntos
cat_cols = ["job", "marital", "education", "contact", "poutcome", "y"]
{col: sorted(bank_raw[col].dropna().unique()) for col in cat_cols}

{'job': ['admin.',
  'blue-collar',
  'entrepreneur',
  'housemaid',
  'management',
  'retired',
  'self-employed',
  'services',
  'student',
  'technician',
  'unemployed'],
 'marital': ['DIVORCED', 'MARRIED', 'SINGLE'],
 'education': ['basic.4y',
  'basic.6y',
  'basic.9y',
  'high.school',
  'illiterate',
  'professional.course',
  'university.degree'],
 'contact': ['cellular', 'telephone'],
 'poutcome': ['FAILURE', 'NONEXISTENT', 'SUCCESS'],
 'y': ['no', 'yes']}

In [13]:
# Columnas numéricas guardadas como texto
bank_raw[["cons.price.idx", "cons.conf.idx", "euribor3m", "nr.employed"]].head()

,cons.price.idx,cons.conf.idx,euribor3m,nr.employed
0,"93,994","-36,4","4,857",5191
1,"93,994","-36,4",NaN,5191
2,"93,994","-36,4","4,857",5191
3,"93,994","-36,4",NaN,5191
4,"93,994","-36,4",NaN,5191


In [14]:
# Duplicados: filas completas, ids y filas idénticas salvo el id
print("Filas duplicadas:", bank_raw.duplicated().sum())
print("IDs duplicados:", bank_raw["id_"].duplicated().sum())
print("Duplicados ignorando índice e id:", bank_raw.drop(columns=["Unnamed: 0", "id_"]).duplicated().sum())

Filas duplicadas: 0
IDs duplicados: 0
Duplicados ignorando índice e id: 0


No hay duplicados. Cada fila es una interacción con un cliente distinto.

### 2.2 Dataset de clientes

In [15]:
customers_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43170 entries, 0 to 43169
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Unnamed: 0         43170 non-null  int64         
 1   Income             43170 non-null  int64         
 2   Kidhome            43170 non-null  int64         
 3   Teenhome           43170 non-null  int64         
 4   Dt_Customer        43170 non-null  datetime64[ns]
 5   NumWebVisitsMonth  43170 non-null  int64         
 6   ID                 43170 non-null  object        
 7   anio_alta          43170 non-null  int64         
dtypes: datetime64[ns](1), int64(6), object(1)
memory usage: 2.6+ MB


In [16]:
print("IDs duplicados:", customers_raw["ID"].duplicated().sum())
customers_raw.describe().round(2)

IDs duplicados: 0


,Unnamed: 0,Income,Kidhome,Teenhome,Dt_Customer,NumWebVisitsMonth,anio_alta
count,43170.00,43170.00,43170.00,43170.00,43170,43170.00,43170.00
mean,7916.02,93227.39,1.00,1.00,2013-05-11 22:09:47.407922176,16.59,2012.86
min,0.00,5841.00,0.00,0.00,2012-01-01 00:00:00,1.00,2012.00
25%,3597.00,49608.00,0.00,0.00,2012-07-16 00:00:00,9.00,2012.00
50%,7194.50,93009.50,1.00,1.00,2013-03-01 00:00:00,17.00,2013.00
75%,11705.75,136740.50,2.00,2.00,2014-03-28 00:00:00,25.00,2014.00
max,20114.00,180802.00,2.00,2.00,2014-12-31 00:00:00,32.00,2014.00
std,5228.25,50498.18,0.82,0.82,NaN,9.24,0.88


El dataset de clientes está mucho más limpio: sin nulos, sin duplicados y con la fecha ya como `datetime`. Solo hay que eliminar `Unnamed: 0`, renombrar columnas y crear variables derivadas.

Comprobamos además que **todas las altas de cada hoja corresponden a su año**:

In [17]:
customers_raw.groupby("anio_alta")["Dt_Customer"].agg(["min", "max", "count"])

,min,max,count
anio_alta,,,
2012,2012-01-01,2012-12-31,20115
2013,2013-01-01,2013-12-31,8965
2014,2014-01-01,2014-12-31,14090


## 3. Limpieza del dataset de campañas

### 3.1 Justificación de la estrategia de nulos

Antes de imputar, se analiza **de qué depende** cada variable con nulos para elegir una imputación fundamentada en lugar de rellenar a ciegas con la media.

In [18]:
# Paso previo: convertir las columnas de coma decimal para poder analizarlas
tmp = bank_raw.copy()
tmp.columns = fn.normalizar_nombres(tmp.columns)
tmp[fn.COLS_COMA_DECIMAL] = tmp[fn.COLS_COMA_DECIMAL].apply(fn.texto_a_float)

grupo_macro = ["emp_var_rate", "cons_conf_idx", "nr_employed"]

# ¿Cuántos IPC distintos hay por contexto macroeconómico?
print("Máx. valores distintos de cons_price_idx por grupo macro:",
      tmp.groupby(grupo_macro)["cons_price_idx"].nunique().max())

# ¿Cuánto varía el euríbor dentro de cada contexto macro?
tmp.groupby(grupo_macro)["euribor3m"].agg(["count", "median", "std"]).round(3)

Máx. valores distintos de cons_price_idx por grupo macro: 1


count  median    std
emp_var_rate cons_conf_idx nr_employed                      
-3.4         -30.1         5017.5         306   0.715  0.003
             -29.8         5017.5         221   0.781  0.023
             -26.9         5017.5         355   0.739  0.009
-3.0         -33.0         5023.5         132   0.714  0.005
-2.9         -40.8         5076.2         574   1.262  0.023
             -33.6         5076.2         139   1.044  0.053
             -31.4         5076.2         655   0.873  0.019
-1.8         -50.0         5099.1         233   1.640  0.101
             -47.1         5099.1        2008   1.405  0.029
             -46.2         5099.1        4744   1.291  0.033
             -40.0         5008.7         186   0.684  0.010
             -34.8         5008.7         224   0.646  0.007
             -34.6         5008.7         145   0.644  0.007
-1.7         -40.3         4991.6         261   0.846  0.037
             -39.8         4991.6         197   0.732  0.020
             -38.3         4991.6         188   0.900  0.005
-1.1         -50.8         4963.6         111   1.044  0.007
             -49.5         4963.6         173   0.987  0.028
             -37.5         4963.6         247   0.879  0.003
-0.2         -45.9         5176.3           7   3.428  0.220
-0.1         -42.0         5195.8        2956   4.120  0.079
             -40.4         5195.8          36   4.860  0.067
 1.1         -36.4         5191.0        6258   4.857  0.002
 1.4         -42.7         5228.1        5467   4.962  0.003
             -41.8         5228.1        3681   4.959  0.044
             -36.1         5228.1        4240   4.964  0.002

- **`cons_price_idx`** (471 nulos): cada combinación de indicadores macro tiene **un único** IPC → la imputación por grupo recupera el **valor exacto**.
- **`euribor3m`** (9.256 nulos, 21,5 %): dentro de cada contexto macro la desviación típica es casi siempre < 0,1 puntos → la **mediana del grupo** es una estimación muy precisa. Eliminar estas filas supondría perder 1 de cada 5 registros.

In [19]:
# La edad depende mucho de la profesión -> imputar con la mediana de su 'job'
tmp.groupby("job")["age"].median().sort_values()

,age
job,
student,25.0
admin.,36.0
services,36.0
technician,37.0
unemployed,38.0
self-employed,39.0
blue-collar,39.0
entrepreneur,41.0
management,42.0


- **`age`** (5.120 nulos, 11,9 %): la mediana va de 25 años (*student*) a 59 (*retired*). Imputar con la mediana global (38) distorsionaría esos grupos, así que se usa la **mediana por profesión** y se guarda un indicador `age_imputada`.
- **`job`, `marital`, `education`**: el nulo se trata como categoría `unknown` (no se inventa información).
- **`default`, `housing`, `loan`**: se dejan como entero *nullable* (`Int8`) con `<NA>` = desconocido. `default` tiene un 21 % de nulos y **solo 3 clientes con valor 1**, por lo que apenas aporta información.
- **`date`** (248 nulos, 0,6 %): no se puede reconstruir; se deja como `NaT`.

### 3.2 Aplicación de la limpieza

In [20]:
bank = fn.limpiar_bank(bank_raw)
print(bank.shape)
bank.head()

(43000, 31)


,age,job,marital,education,default,housing,loan,contact,duration,campaign,pdays,previous,poutcome,emp_var_rate,cons_price_idx,cons_conf_idx,euribor3m,nr_employed,y,date,latitude,longitude,id_cliente,contact_year,contact_month,age_imputada,contactado_antes,y_num,duration_min,grupo_edad,grupo_campaign
0,45,housemaid,married,basic_4y,0,0,0,telephone,261,1,NaN,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no,2019-08-02,41.495,-71.233,089b39d8-e4d0-461b-87d4-814d71e0e079,2019,8,1,0,0,4.35,45-54,1
1,57,services,married,high_school,<NA>,0,0,telephone,149,1,NaN,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no,2016-09-14,34.601,-83.923,e9d37224-cb6f-4942-98d7-46672963d097,2016,9,0,0,0,2.48,55-64,1
2,37,services,married,high_school,0,1,0,telephone,226,1,NaN,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no,2019-02-15,34.939,-94.847,3f9f49b5-e410-4948-bf6e-f9244f04918b,2019,2,0,0,0,3.77,35-44,1
3,40,admin,married,basic_6y,0,0,0,telephone,151,1,NaN,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no,2015-11-29,49.041,-70.308,9991fafb-4447-451a-8be2-b0df6098d13e,2015,11,0,0,0,2.52,35-44,1
4,56,services,married,high_school,0,0,1,telephone,307,1,NaN,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no,2017-01-29,38.033,-104.463,eca60b76-70b6-4077-80ba-bc52e8ebb0eb,2017,1,0,0,0,5.12,55-64,1


In [21]:
fn.resumen_calidad(bank)

,tipo,nulos,pct_nulos,valores_unicos
pdays,float64,41412,96.31,26
default,Int8,8981,20.89,2
housing,Int8,1026,2.39,2
loan,Int8,1026,2.39,2
contact_month,Int8,248,0.58,12
contact_year,Int16,248,0.58,5
date,datetime64[ns],248,0.58,1825
education,category,0,0.00,8
contact,category,0,0.00,2
duration,int16,0,0.00,1540


Los únicos nulos que quedan son intencionados: `pdays` (clientes nunca contactados), las binarias desconocidas y las 248 fechas ausentes.

In [22]:
# Verificaciones automáticas de la limpieza
comprobaciones = {
    "sin nulos en age": bank["age"].notna().all(),
    "sin nulos en euribor3m": bank["euribor3m"].notna().all(),
    "sin nulos en cons_price_idx": bank["cons_price_idx"].notna().all(),
    "categorías en minúscula": all(bank[c].astype(str).str.islower().all() for c in ["marital", "poutcome"]),
    "edad en rango 17-98": bank["age"].between(17, 98).all(),
    "pdays coherente con poutcome": (bank.loc[bank["contactado_antes"] == 1, "poutcome"] != "nonexistent").all(),
    "mismas filas que el original": len(bank) == len(bank_raw),
}
for nombre, ok in comprobaciones.items():
    print(f"{'OK ' if ok else 'ERROR'} {nombre}")
assert all(comprobaciones.values())

OK  sin nulos en age
OK  sin nulos en euribor3m
OK  sin nulos en cons_price_idx
OK  categorías en minúscula
OK  edad en rango 17-98
OK  pdays coherente con poutcome
OK  mismas filas que el original


### 3.3 Coherencia lógica entre columnas

In [23]:
# previous = 0 debe implicar poutcome = 'nonexistent'
pd.crosstab(bank["previous"] > 0, bank["poutcome"])

poutcome,failure,nonexistent,success
previous,,,
False,0,37103,0
True,4461,0,1436


La relación es coherente: todos los clientes sin contactos previos tienen `poutcome = nonexistent`.

In [24]:
# Valores de las nuevas columnas de fecha
for col in ["contact_year", "contact_month"]:
    conteo = bank[col].value_counts().sort_index()
    print(col, {int(k): int(v) for k, v in conteo.items()})

contact_year {2015: 8544, 2016: 8533, 2017: 8562, 2018: 8549, 2019: 8564}
contact_month {1: 3517, 2: 3577, 3: 3589, 4: 3582, 5: 3547, 6: 3538, 7: 3593, 8: 3526, 9: 3569, 10: 3599, 11: 3603, 12: 3512}


⚠️ **Inconsistencia detectada en `date`**: las fechas están repartidas de forma casi perfectamente uniforme entre 2015 y 2019 y entre los 12 meses. Sin embargo, el euríbor, el IPC o el nº de empleados son **indicadores mensuales**: en un mismo mes deberían tomar un único valor. Veamos cuántos valores distintos de euríbor hay en cada mes:

In [25]:
bank.groupby(["contact_year", "contact_month"])["euribor3m"].nunique().describe().round(1)

,euribor3m
count,60.0
mean,109.1
std,6.1
min,97.0
25%,105.0
50%,109.0
75%,113.2
max,122.0


Cada mes tiene ~110 valores distintos de euríbor (de 0,6 % a 5 %). Conclusión: **la columna `date` no es coherente con el contexto macroeconómico** (probablemente se generó de forma aleatoria). Se conserva, pero en el análisis el "momento" real de la campaña se medirá a través de los indicadores macro, no de la fecha.

In [26]:
# Las coordenadas no están descritas en el enunciado: comprobamos su rango
bank[["latitude", "longitude"]].describe().loc[["min", "max"]]

,latitude,longitude
min,24.396,-124.997
max,49.384,-66.937


Latitud 24–49 y longitud −125 a −67 corresponden a **Estados Unidos**, no a Portugal. No están documentadas y son incoherentes con el contexto, así que **no se usarán en el análisis** (se conservan en el dataset por trazabilidad).

### 3.4 Valores atípicos

Se detectan con el criterio IQR (1,5·IQR).

In [27]:
cols_outliers = ["age", "duration", "campaign", "previous"]
pd.DataFrame({
    "n_outliers": {c: fn.outliers_iqr(bank[c]).sum() for c in cols_outliers},
    "pct": {c: round(fn.outliers_iqr(bank[c]).mean() * 100, 2) for c in cols_outliers},
    "max": {c: bank[c].max() for c in cols_outliers},
})

,n_outliers,pct,max
age,576,1.34,98
duration,3072,7.14,4918
campaign,2504,5.82,56
previous,5897,13.71,7


Los atípicos **no son errores**: son comportamientos reales (llamadas largas de hasta 82 min, clientes de hasta 98 años o contactados 56 veces). Eliminarlos haría perder justo los casos más informativos (por ejemplo, las llamadas largas son las que más convierten). En `previous` cualquier valor > 0 aparece como atípico porque el 86 % de los clientes nunca fue contactado antes. Se **mantienen** y, en el análisis, se usan medianas y agrupaciones (`grupo_campaign`) que son robustas frente a ellos.

Solo hay 4 llamadas con `duration = 0`; se mantienen (llamadas no atendidas).

## 4. Limpieza del dataset de clientes

In [28]:
customers = fn.limpiar_customers(customers_raw)
customers.head()

,income,kidhome,teenhome,dt_customer,num_web_visits_month,id_cliente,anio_alta,hijos_total,tiene_hijos,grupo_ingresos,grupo_web
0,161770,1,0,2012-04-04,29,089b39d8-e4d0-461b-87d4-814d71e0e079,2012,1,1,Q4 alto,>20
1,85477,1,1,2012-12-30,7,e9d37224-cb6f-4942-98d7-46672963d097,2012,2,1,Q2,6-10
2,147233,1,1,2012-02-02,5,3f9f49b5-e410-4948-bf6e-f9244f04918b,2012,2,1,Q4 alto,0-5
3,121393,1,2,2012-12-21,29,9991fafb-4447-451a-8be2-b0df6098d13e,2012,3,1,Q3,>20
4,63164,1,2,2012-06-20,20,eca60b76-70b6-4077-80ba-bc52e8ebb0eb,2012,3,1,Q2,11-20


In [29]:
comprobaciones = {
    "sin nulos": customers.notna().all().all(),
    "ids únicos": customers["id_cliente"].is_unique,
    "ingresos positivos": (customers["income"] > 0).all(),
    "hijos en rango 0-2": customers[["kidhome", "teenhome"]].isin([0, 1, 2]).all().all(),
    "año de alta coherente con la fecha": (customers["dt_customer"].dt.year == customers["anio_alta"]).all(),
}
for nombre, ok in comprobaciones.items():
    print(f"{'OK ' if ok else 'ERROR'} {nombre}")
assert all(comprobaciones.values())

OK  sin nulos
OK  ids únicos
OK  ingresos positivos
OK  hijos en rango 0-2
OK  año de alta coherente con la fecha


## 5. Unión de los datasets

La clave común es el identificador del cliente (`id_` en el CSV, `ID` en el Excel, renombrados a `id_cliente`). Primero se comprueba la cobertura:

In [30]:
ids_bank = set(bank["id_cliente"])
ids_cust = set(customers["id_cliente"])
print(f"Clientes en campañas:            {len(ids_bank):,}")
print(f"Clientes en el Excel:            {len(ids_cust):,}")
print(f"En ambos:                        {len(ids_bank & ids_cust):,}")
print(f"Solo en el Excel (sin contacto): {len(ids_cust - ids_bank):,}")

Clientes en campañas:            43,000
Clientes en el Excel:            43,170
En ambos:                        43,000
Solo en el Excel (sin contacto): 170


Todos los clientes contactados tienen ficha. Se hace un `left join` desde campañas (la unidad de análisis es la interacción) con `validate='one_to_one'` para garantizar que no se duplican filas.

In [31]:
df = bank.merge(customers, on="id_cliente", how="left", validate="one_to_one")
print(df.shape)
assert len(df) == len(bank) and df["income"].notna().all()

# Coherencia temporal: nadie debería ser contactado antes de hacerse cliente
print("Contactos anteriores al alta:", (df["date"] < df["dt_customer"]).sum())

(43000, 41)
Contactos anteriores al alta: 0


## 6. Optimización de memoria y exportación

In [32]:
memoria = {
    "bank (bruto)": fn.memoria_mb(bank_raw),
    "bank (limpio)": fn.memoria_mb(bank),
    "customers (bruto)": fn.memoria_mb(customers_raw),
    "customers (limpio)": fn.memoria_mb(customers),
    "dataset final": fn.memoria_mb(df),
}
pd.Series(memoria, name="MB")

,MB
bank (bruto),32.74
bank (limpio),7.71
customers (bruto),5.81
customers (limpio),4.36
dataset final,8.58


Convertir las variables de texto repetitivo a `category` y reducir los enteros (`int8`, `int16`) **reduce la memoria del dataset de campañas a menos de una cuarta parte**.

In [33]:
fn.RUTA_PROCESSED.mkdir(parents=True, exist_ok=True)
salidas = {"bank_clean.csv": bank, "customers_clean.csv": customers, "bank_customers_clean.csv": df}
for nombre, datos in salidas.items():
    datos.to_csv(fn.RUTA_PROCESSED / nombre, index=False)
    print(f"Guardado data/processed/{nombre}  ({len(datos):,} filas)")

Guardado data/processed/bank_clean.csv  (43,000 filas)
Guardado data/processed/customers_clean.csv  (43,170 filas)
Guardado data/processed/bank_customers_clean.csv  (43,000 filas)


### Resumen de la limpieza

- **43.000** interacciones y **43.170** clientes; sin duplicados.
- 4 columnas numéricas convertidas de texto con coma decimal a `float`.
- Categorías homogeneizadas (minúsculas, sin puntos) y fecha en español parseada; creadas `contact_month` y `contact_year`.
- Nulos imputados con criterio (edad por profesión; euríbor e IPC por contexto macro) o mantenidos como `unknown` / `<NA>` cuando no hay base para imputar.
- `pdays = 999` recodificado a `NaN` + indicador `contactado_antes`.
- Variables nuevas: `y_num`, `duration_min`, `grupo_edad`, `grupo_campaign`, `hijos_total`, `tiene_hijos`, `grupo_ingresos`, `grupo_web`, `anio_alta`.
- Detectadas dos incoherencias en los datos de origen: `date` no encaja con los indicadores macro y las coordenadas son de EE. UU.
- Unión 1:1 sin pérdida de registros y memoria reducida ~75 %.